Verify GPU Is **Enabled**

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


**Install Required Libraries**

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate scikit-learn indic-nlp-library


**Verify Installations**

In [ ]:
import sklearn
import torch
import transformers
import datasets

print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)


**Load using Pandas**

In [ ]:
import pandas as pd

# Change filename if needed
df = pd.read_csv("train.csv")

# View first 5 rows
df.head()


**Basic Sanity Checks**

In [ ]:
df.shape


In [ ]:
df.columns

In [ ]:

df.isnull().sum()

**Label Distribution Analysis**

In [ ]:
df['labels'].value_counts()


**Verify Text Quality**

In [ ]:
df.sample(5)[['content', 'labels']]


**Label Encoding**

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
dev_df   = pd.read_csv("dev.csv")



In [ ]:
label2id = {
    "Substantiated": 0,
    "Sarcastic": 1,
    "Opinionated": 2,
    "Positive": 3,
    "Negative": 4,
    "Neutral": 5,
    "None of the above": 6
}

id2label = {v: k for k, v in label2id.items()}

for df in [train_df, dev_df]:
    df['label_id'] = df['labels'].map(label2id)

train_df[['labels', 'label_id']].head(), dev_df[['labels', 'label_id']].head()




**TEXT PREPROCESSING**
Common Preprocessing (Applied to Both)

In [ ]:
import re
import unicodedata

#Unicode Normalization (Tamil-safe)
def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

#Remove URLs
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

#Remove Mentions (@username)
def remove_mentions(text):
    return re.sub(r"@\w+", "", text)
#Basic Cleanup Function
def basic_preprocess(text):
    text = normalize_unicode(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = text.strip()
    return text
train_df['clean_text'] = train_df['content'].astype(str).apply(basic_preprocess)
dev_df['clean_text']   = dev_df['content'].astype(str).apply(basic_preprocess)

train_df[['content', 'clean_text']].head(), dev_df[['content', 'clean_text']].head()




**Baseline Preprocessing (TF-IDF + SVM)**

In [ ]:
#Remove Emojis & Special Symbols
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub(r"", text)
#Remove Extra Punctuation & Numbers
def remove_special_chars(text):
    return re.sub(r"[^0-9A-Za-z\u0B80-\u0BFF\s]", "", text)
#Lowercasing (Safe for Tamil)
def lowercase(text):
    return text.lower()
#Final Baseline Pipeline
def preprocess_for_tfidf(text):
    text = basic_preprocess(text)
    text = remove_emojis(text)
    text = remove_special_chars(text)
    text = lowercase(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_df['tfidf_text'] = train_df['content'].astype(str).apply(preprocess_for_tfidf)
dev_df['tfidf_text']   = dev_df['content'].astype(str).apply(preprocess_for_tfidf)

train_df[['content', 'tfidf_text']].head(), dev_df[['content', 'tfidf_text']].head()


**Transformer Preprocessing (XLM-RoBERTa)**

In [ ]:
#Minimal Cleaning Only
def preprocess_for_transformer(text):
    text = basic_preprocess(text)
    return text

train_df['xlmr_text'] = train_df['content'].astype(str).apply(preprocess_for_transformer)
dev_df['xlmr_text']   = dev_df['content'].astype(str).apply(preprocess_for_transformer)

train_df[['content', 'xlmr_text']].head(), dev_df[['content', 'xlmr_text']].head()




In [ ]:
#Verify Preprocessing Differences (Reviewer-friendly)
train_df.sample(5)[['content', 'tfidf_text', 'xlmr_text']]



In [ ]:
#Store Processed Data (Optional but Clean)
train_df[['tfidf_text', 'xlmr_text', 'label_id']].to_csv(
    "processed_train.csv",
    index=False
)

dev_df[['tfidf_text', 'xlmr_text', 'label_id']].to_csv(
    "processed_dev.csv",
    index=False
)

**BASELINE MODEL — TF-IDF + SVM**

In [ ]:
#Prepare Train / Validation
X_train = train_df['tfidf_text']
y_train = train_df['label_id']

X_val = dev_df['tfidf_text']
y_val = dev_df['label_id']




#Create TF-IDF Vectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)



In [ ]:
#Train the Model
from sklearn.svm import LinearSVC

svm = LinearSVC(
    C=1.0,
    class_weight='balanced',
    random_state=42
)

svm.fit(X_train_tfidf, y_train)
#Validate the Model

from sklearn.metrics import classification_report

y_val_pred = svm.predict(X_val_tfidf)

print(classification_report(
    y_val,
    y_val_pred,
    target_names=[id2label[i] for i in range(7)],
    digits=4
))




In [ ]:
import pandas as pd

test_df = pd.read_csv("test.csv")
test_df.head()

In [ ]:
# 1. Preprocess test text
test_df['tfidf_text'] = test_df['content'].astype(str).apply(preprocess_for_tfidf)

# 2. Transform using trained TF-IDF vectorizer
X_test = test_df['tfidf_text']
X_test_tfidf = tfidf.transform(X_test)

# 3. Predict label IDs
test_preds = svm.predict(X_test_tfidf)

# 4. Convert IDs → label names
test_df['predicted_label'] = [id2label[i] for i in test_preds]

# 5. Save submission file with content + prediction
test_df[['content', 'predicted_label']].to_csv("baseline_submission.csv", index=False)

print("✅ Saved: baseline_submission.csv")



In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val, y_val_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[id2label[i] for i in range(7)]
)

plt.figure(figsize=(8, 6))
disp.plot(values_format='d')
plt.title("TF-IDF + SVM Validation Confusion Matrix")
plt.show()



**TRANSFORMER MODEL — XLM-RoBERTa**

In [ ]:

# 1. Imports

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)


# 2. Load Data

train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("dev.csv")
test_df  = pd.read_csv("test.csv")


# 3. Text Preprocessing

def preprocess_xlmr(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["xlmr_text"] = train_df["content"].apply(preprocess_xlmr)
val_df["xlmr_text"]   = val_df["content"].apply(preprocess_xlmr)
test_df["xlmr_text"]  = test_df["content"].apply(preprocess_xlmr)


# 4. Label Encoding

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df["labels"])
val_labels   = label_encoder.transform(val_df["labels"])

id2label = dict(enumerate(label_encoder.classes_))
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)


# 5. Tokenizer

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# 6. Dataset Classes

class XLMRDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze() for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class XLMRTestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze() for k, v in encoding.items()}


# 7. Datasets

train_dataset = XLMRDataset(
    train_df["xlmr_text"].tolist(),
    train_labels,
    tokenizer
)

val_dataset = XLMRDataset(
    val_df["xlmr_text"].tolist(),
    val_labels,
    tokenizer
)


# 8. Class Weights (CRITICAL)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# 9. Model

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# 10. Custom Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


# 11. Metrics

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }


# 12. Training Arguments

training_args = TrainingArguments(
    output_dir="./xlmr_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=6,
    warmup_ratio=0.1,
    weight_decay=0.01,

    logging_steps=50,
    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)

# 13. Trainer

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 14. Train

trainer.train()


# 15. Evaluate

eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)


# 16. Predict on Test Set

test_dataset = XLMRTestDataset(
    test_df["xlmr_text"].tolist(),
    tokenizer
)

test_preds = trainer.predict(test_dataset)
test_label_ids = np.argmax(test_preds.predictions, axis=1)

test_df["predicted_label"] = [
    id2label[i] for i in test_label_ids
]


# 17. Save Submission

test_df[["content", "predicted_label"]].to_csv(
    "xlmr_submission.csv",
    index=False
)

print("✅ Saved submission: xlmr_submission.csv")

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

In [ ]:
#confusion matrix heatmap
import seaborn as sns
plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=list(id2label.values()),
    yticklabels=list(id2label.values())
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
#Smoothed Training Loss Curve
import numpy as np

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:
    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

# Smooth curve
window = 2
smoothed = np.convolve(train_loss, np.ones(window)/window, mode='valid')

plt.figure(figsize=(7,5))
plt.plot(epochs[:len(smoothed)], smoothed)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Smoothed Training Loss Curve")
plt.show()

In [ ]:
#Validation F1 Progress Graph
eval_macro = []
eval_weighted = []
epochs = []

for log in logs:
    if "eval_macro_f1" in log:
        eval_macro.append(log["eval_macro_f1"])
        eval_weighted.append(log["eval_weighted_f1"])
        epochs.append(log["epoch"])

plt.figure(figsize=(7,5))
plt.plot(epochs, eval_macro, label="Macro F1")
plt.plot(epochs, eval_weighted, label="Weighted F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Validation F1 Progress")
plt.legend()
plt.show()